# NLP TRIPADVISOR PROJECT BY BOULMIER ILAN

## 1 DATA CLEANING

### 1.1 Import

In [1]:
import pandas as pd 
import numpy as np
pd.set_option('display.max_colwidth', None)

In [2]:
reviews = pd.read_csv("reviews83325.csv")
places = pd.read_csv("Tripadvisor.csv")

C:\Users\ilanb\AppData\Local\Temp\ipykernel_4396\3059705544.py:1: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv("reviews83325.csv")


In [3]:
print(reviews.shape)
print(places.shape)

(340385, 21)
(3761, 60)


### 1.2 English filter

In [4]:
reviews['langue'].value_counts()

langue
en      153071
fr       99078
it       22912
es       21768
pt       19521
ru        5463
de        5237
ja        4124
nl        2835
ko        1133
sv         920
da         666
el         570
tr         570
pl         490
zhTW       488
no         426
zhCN       408
iw         185
cs          81
th          81
ar          78
fi          76
hu          73
in          61
sr          31
sk          29
vi          10
Name: count, dtype: int64

In [5]:
reviews_en = reviews[reviews["langue"] == "en"].copy()

print(reviews_en.shape)

(153071, 21)


### 1.3 Concatenation by place id

by place, we concatenate all the corresponding reviews

In [6]:
grouped_reviews = (
    reviews_en
    .groupby("idplace")["review"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

In [7]:
places_with_reviews = places.merge(
    grouped_reviews,
    left_on="id",
    right_on="idplace",
    how="inner"
)

In [8]:
print("Nombre de lieux distincts :", reviews_en["idplace"].nunique())

Nombre de lieux distincts : 1835


### 1.4 TRAIN/TEST 50%

In [9]:
from sklearn.model_selection import train_test_split

train_places, test_places = train_test_split(
    places_with_reviews,
    test_size=0.5,
    random_state=42
)

print("Train:", train_places.shape[0])
print("Test:", test_places.shape[0])

Train: 917
Test: 918


Verification intersection train/test

In [21]:
set(train_places["id"]).intersection(set(test_places["id"]))

set()

## 2 Model + Evaluation

### 2.1.1 BM25 Baseline

In [28]:
!pip install rank-bm25

In [29]:
from rank_bm25 import BM25Okapi

In [30]:
# On récupère les textes du test
test_texts = test_places["review"].tolist()

# Tokenisation simple
test_tokens = [doc.lower().split() for doc in test_texts]

In [32]:
bm25 = BM25Okapi(test_tokens)

Here you can choose de query by modifying the index in the query_text variable. We took 0 so the first review in our train dataset as an exemple.

In [37]:
query_text = train_places.iloc[0]["review"]
query_tokens = query_text.lower().split()

scores = bm25.get_scores(query_tokens)

this score represent the best representative document link to the query. If the score is high, then this place is a good candidate.

In [44]:
print(scores)

[ 867.56171176 1204.30336297  398.13726364  435.19464224  779.50602484
  728.70941877  553.39960894 1209.33780412  345.10046105 1143.0520231
  771.48483458  761.60429091  992.46367111 1017.60387704  975.09881796
  579.34697864  519.33814511  742.64946346  997.07347679  974.71465848
  892.42586835  788.00732113  515.51479971  524.16106267 1175.5488186
 1263.56052583 1054.98844019  915.38408978 1188.6895721   349.5332286
  442.89763924 1204.65020793  668.17390644  840.13638787 1188.29700943
 1120.13442346 1106.43226017  486.39931588  248.8290513   645.25587004
  484.07237456  794.30481343 1195.81524484  639.1670243  1060.6523269
  722.89757601  407.72659558  993.97122103  961.84830259  891.00816867
  339.00209385  977.30099361  699.0498927  1102.8626411  1231.47112563
 1200.32024824  916.4408533   676.80297412  474.8954174  1045.44750223
 1241.95923587  812.60707816  965.7184012   804.87453764 1194.22452579
  733.17208684 1389.50018866 1124.18167292 1245.71910854  606.13594052
 1004.3987

In [40]:
ranking = np.argsort(scores)[::-1]  # tri décroissant

here we are printing the index in the place list. Then The place 236 in my list is the best one for the query

In [45]:
print(ranking)

[236 816 377  66 147 516 644 638 817 726 292 351 134 446  99  86 273 154
 785 598 269 685 578 232 185 557 307 658  89 443 762 355 605 362 347 601
 253 732 684 486 495 501  25 864  87 823 535 696 833 104  77 367 490 340
  68 706 589 567 868  60 842 836 438 283 503 709 634  54 505 395 621 233
 193 877 626 485 739 562 686 869 694 454 206 170 613 396 774 453 819 784
 349 220 695 548 719   7 815 810 513 437  31   1 906 366 188 469 770 782
 901  55 390 611 289 773 731  42 609 710  64 903 671 665 457 916 799  80
  28 740 137  34 450 668 376 475 240 492 661 353 433 569 494 107 779  24
 159 636 844 393 348 853 777 270 317 867 537 436 783 242 493 146 866 262
 659 797 315 655 408 105 897 610 247 123 737 893 714 822 225 165 750 148
 213 295 268 500 749   9 760 218 413 109 234 879 173 736 400 293 756 299
 222 424 192 529 251 274 426  97 539 917  67 324 811 420  73  35 439 430
 434 911 461 565 531 161 863  36 597 156 226  53 761 407 181  96 229 865
 112 378 191 419 649 697 874 813 841 873  81 701 24

In [42]:
best_index = ranking[0]
best_place = test_places.iloc[best_index]

print("Query type:", train_places.iloc[0]["typeR"])
print("Recommended type:", best_place["typeR"])

Query type: R
Recommended type: R


We want first to see if the recommanded place and the query have the same type so im just printing the type

### 2.1.2 Ranking Evaluation level 1

We look at the first type corresponding place (H,R,A,AP)

In [46]:
import numpy as np
from tqdm import tqdm  # barre de progression

errors = []

for i in tqdm(range(len(train_places))):
    
    query_text = train_places.iloc[i]["review"]
    query_type = train_places.iloc[i]["typeR"]
    
    query_tokens = query_text.lower().split()
    
    scores = bm25.get_scores(query_tokens)
    ranking = np.argsort(scores)[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_type = test_places.iloc[idx]["typeR"]
        
        if candidate_type == query_type:
            error = rank
            break
    
    if error is not None:
        errors.append(error)

100%|██████████| 917/917 [1:22:27<00:00,  5.39s/it]  


printing the list of the first index where the candidate type and query type are the same for each review in our test dataset and after we will print the average rank in order to evaluate our model. By the way, we will do this for each model we going to test in this project

In [51]:
print(errors)

[0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 4, 0, 0, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 3, 1, 1, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 3, 10, 0, 0, 0, 1, 30, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 27, 0, 1, 2, 0, 1, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 3, 0, 0, 0, 2, 1, 0, 0, 0, 1, 0, 1, 15, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 2, 0, 0, 2, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 3, 4, 0, 0, 0, 0, 0, 0, 0, 1, 3, 0, 1, 0, 0, 0, 0, 9, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 3, 0, 1, 0, 7, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 12, 0, 0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 5, 0, 0, 0, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 4, 1, 0, 0, 0, 0, 2, 2, 5, 1, 15, 0, 1, 21, 0, 1, 0, 0, 0, 11, 1, 1, 1, 0, 0, 1, 0, 0, 2, 0, 0, 0, 1, 0, 3, 0, 8, 1, 4, 0, 0, 0, 17, 33, 0, 1, 0, 0, 0, 0, 3, 1, 1, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 1, 4, 0, 0, 2, 4, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 1, 1, 0,

In [47]:
mean_error = np.mean(errors)
print("Ranking Error Level 1 (BM25):", mean_error)

Ranking Error Level 1 (BM25): 0.9389312977099237


In [48]:
print("Number of evaluated queries:", len(errors))
print("Total queries:", len(train_places))

Number of evaluated queries: 917
Total queries: 917


### 2.2.1 TF/IDF + cosine similarity

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [53]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000
)

In [54]:
test_texts = test_places["review"].tolist()

tfidf_test = vectorizer.fit_transform(test_texts)

In [55]:
query_text = train_places.iloc[0]["review"]

tfidf_query = vectorizer.transform([query_text])

similarities = cosine_similarity(tfidf_query, tfidf_test)[0]

ranking = similarities.argsort()[::-1]

In [57]:
best_index = ranking[0]
best_place = test_places.iloc[best_index]

print("Query type:", train_places.iloc[0]["typeR"])
print("Recommended type:", best_place["typeR"])

Query type: R
Recommended type: R


### 2.2.2 Evaluation ranking level 1 

In [58]:
errors_tfidf = []

for i in tqdm(range(len(train_places))):
    
    query_text = train_places.iloc[i]["review"]
    query_type = train_places.iloc[i]["typeR"]
    
    tfidf_query = vectorizer.transform([query_text])
    similarities = cosine_similarity(tfidf_query, tfidf_test)[0]
    
    ranking = similarities.argsort()[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_type = test_places.iloc[idx]["typeR"]
        
        if candidate_type == query_type:
            error = rank
            break
    
    if error is not None:
        errors_tfidf.append(error)

100%|██████████| 917/917 [00:41<00:00, 22.03it/s]


In [59]:
mean_error_tfidf = np.mean(errors_tfidf)

print("Ranking Error Level 1 (TF-IDF):", mean_error_tfidf)

Ranking Error Level 1 (TF-IDF): 0.8898582333696837


In [60]:
print("BM25:", mean_error)
print("TF-IDF:", mean_error_tfidf)

BM25: 0.9389312977099237
TF-IDF: 0.8898582333696837


### 2.3 Evaluation ranking level 2

In [61]:
places_with_reviews.columns

Index(['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis',
       'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse',
       'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars',
       'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete',
       'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance',
       'hotelbearing', 'restaurantTypeCuisine',
       'restaurantDietaryRestrictions', 'restaurantMeals',
       'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService',
       'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType',
       'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview',
       'dateLastScanReviews', 'shape_gid', 'gadm36_gid', 'hotelprice',
       'hotelBookingID', 'restaurantSubcategory', 'restaurantType',
       'ap_additional_info', 'ap_age_band_list', 'ap_attraction_ids',
       'ap_booking_question_list', 'ap_bubble_rating_integer', 'ap_duration',
       'ap_exclusion', '

In [89]:
def safe_split(value):
    "intersection verification"""
    if pd.isna(value):
        return set()
    return set(str(value).lower().split(","))

here is our main function on metadata level 2 evaluation. I choose the best metadatas, not too much because we could overfit our model and make it useless. if we choose too much metadata the model will never find a coresponding place because a lot of criteria are given.

In [88]:
def is_match_level2(query_row, candidate_row):
    """we are matching on metada for each type of places"""
    query_type = query_row["typeR"]
    
    # HOTEL
    if query_type == "H":
        return query_row["priceRange"] == candidate_row["priceRange"]
    
    # RESTAURANT
    elif query_type == "R":
        q_types = safe_split(query_row["restaurantType"])
        c_types = safe_split(candidate_row["restaurantType"])
        
        q_cuisine = safe_split(query_row["restaurantTypeCuisine"])
        c_cuisine = safe_split(candidate_row["restaurantTypeCuisine"])
        
        return (
            len(q_types & c_types) > 0 or
            len(q_cuisine & c_cuisine) > 0
        )
    
    # ATTRACTION
    elif query_type == "A":
        q_cat = safe_split(query_row["activiteSubCategorie"])
        c_cat = safe_split(candidate_row["activiteSubCategorie"])
        
        q_sub = safe_split(query_row["activiteSubType"])
        c_sub = safe_split(candidate_row["activiteSubType"])
        
        return (
            len(q_cat & c_cat) > 0 or
            len(q_sub & c_sub) > 0
        )
    
    # ATTRACTION PRODUCT
    elif query_type == "AP":
        return query_row["ap_primary_supplier_subtype"] == candidate_row["ap_primary_supplier_subtype"]
    
    return False

### 2.3.1 Evaluation ranking level 2 TF/IDF cosine similarity

In [64]:
errors_level2 = []

for i in tqdm(range(len(train_places))):
    
    query_row = train_places.iloc[i]
    query_text = query_row["review"]
    
    tfidf_query = vectorizer.transform([query_text])
    similarities = cosine_similarity(tfidf_query, tfidf_test)[0]
    
    ranking = similarities.argsort()[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_row = test_places.iloc[idx]
        
        if is_match_level2(query_row, candidate_row):
            error = rank
            break
    
    if error is not None:
        errors_level2.append(error)

100%|██████████| 917/917 [00:49<00:00, 18.65it/s]


In [65]:
mean_error_level2 = np.mean(errors_level2)

print("Ranking Error Level 2 (TF-IDF):", mean_error_level2)

Ranking Error Level 2 (TF-IDF): 3.7733773377337734


In [66]:
print("Evaluated queries:", len(errors_level2))
print("Total queries:", len(train_places))

Evaluated queries: 909
Total queries: 917


In [67]:
train_places["restaurantTypeCuisine"].dropna().iloc[0]

'5086,10654,10665,10992'

### 2.3.2 Evaluation ranking level 2 BM25

In [68]:
errors_level2_bm25 = []

for i in tqdm(range(len(train_places))):
    
    query_row = train_places.iloc[i]
    query_text = query_row["review"]
    
    query_tokens = query_text.lower().split()
    
    scores = bm25.get_scores(query_tokens)
    
    ranking = np.argsort(scores)[::-1]
    
    error = None
    
    # Parcours du ranking
    for rank, idx in enumerate(ranking):
        
        candidate_row = test_places.iloc[idx]
        
        if is_match_level2(query_row, candidate_row):
            error = rank
            break
    
    if error is not None:
        errors_level2_bm25.append(error)

100%|██████████| 917/917 [1:24:49<00:00,  5.55s/it]  


In [69]:
mean_error_level2_bm25 = np.mean(errors_level2_bm25)

print("Ranking Error Level 2 (BM25):", mean_error_level2_bm25)
print("Evaluated queries:", len(errors_level2_bm25))
print("Total queries:", len(train_places))

Ranking Error Level 2 (BM25): 5.666666666666667
Evaluated queries: 909
Total queries: 917


### 2.4 TF/IDF Improved

Lets try here to modify some preliminaries parameters (choices are explained in my report).

In [47]:
vectorizer_improved = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    max_df=0.8
)

In [48]:
tfidf_test_improved = vectorizer_improved.fit_transform(test_places["review"])

We can try to type our reviews to see.

In [66]:
query_text = "I love this attracion park."

tfidf_query = vectorizer_improved.transform([query_text])

similarities = cosine_similarity(tfidf_query, tfidf_test_improved)[0]

ranking = similarities.argsort()[::-1]

In [68]:
best_index = ranking[0]
best_place = test_places.iloc[best_index]


print("Recommended type:", best_place["typeR"])

Recommended type: AP


### 2.4.1 Evaluation ranking level 2

In [28]:
from tqdm import tqdm

errors_level2_tfidf_improved = []

for i in tqdm(range(len(train_places))):
    
    query_row = train_places.iloc[i]
    query_text = query_row["review"]
    
    tfidf_query = vectorizer_improved.transform([query_text])
    similarities = cosine_similarity(tfidf_query, tfidf_test_improved)[0]
    
    ranking = similarities.argsort()[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_row = test_places.iloc[idx]
        
        if is_match_level2(query_row, candidate_row):
            error = rank
            break
    
    if error is not None:
        errors_level2_tfidf_improved.append(error)

100%|██████████| 917/917 [01:00<00:00, 15.04it/s]


In [29]:
mean_error_level2_tfidf_improved = np.mean(errors_level2_tfidf_improved)

print("Ranking Error Level 2 (TF-IDF improved):", mean_error_level2_tfidf_improved)
print("Evaluated queries:", len(errors_level2_tfidf_improved))

Ranking Error Level 2 (TF-IDF improved): 3.3751375137513753
Evaluated queries: 909


### 2.5 New Model test

here I wanted to try a model that I had seen during my practical lesson. This is not the best.

In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

C:\Users\ilanb\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
model = SentenceTransformer("all-MiniLM-L6-v2")

C:\Users\ilanb\anaconda3\envs\myenv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ilanb\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 395.88it/s, Materializing param=pooler

In [33]:
test_texts = test_places["review"].tolist()

test_embeddings = model.encode(
    test_texts,
    show_progress_bar=True,
    batch_size=32
)

Batches: 100%|██████████| 29/29 [01:14<00:00,  2.58s/it]


### 2.5.1 Evaluation ranking level 2 new model

In [34]:
errors_level2_embeddings = []

for i in tqdm(range(len(train_places))):
    
    query_row = train_places.iloc[i]
    query_text = query_row["review"]
    
    query_embedding = model.encode([query_text])
    
    similarities = cosine_similarity(query_embedding, test_embeddings)[0]
    
    ranking = similarities.argsort()[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_row = test_places.iloc[idx]
        
        if is_match_level2(query_row, candidate_row):
            error = rank
            break
    
    if error is not None:
        errors_level2_embeddings.append(error)

100%|██████████| 917/917 [01:55<00:00,  7.94it/s]


In [36]:
mean_error_level2_embeddings = np.mean(errors_level2_embeddings)

print("Ranking Error Level 2 (Embeddings):", mean_error_level2_embeddings)
print("Evaluated queries:", len(errors_level2_embeddings))

Ranking Error Level 2 (Embeddings): 4.240924092409241
Evaluated queries: 909


### 2.6 New test : Max reviews limitations

Instead of take all reviews for one place lets choose 20 for all of them. This avoids giving undue importance to certain places that have a lot of reviews and big length.

### 2.6.1 20 reviews

In [69]:
MAX_REVIEWS = 20

balanced_reviews = (
    reviews_en
    .groupby("idplace")
    .head(MAX_REVIEWS)
)

grouped_balanced = (
    balanced_reviews
    .groupby("idplace")["review"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

In [70]:
places_with_reviews_balanced = places.merge(
    grouped_balanced,
    left_on="id",
    right_on="idplace",
    how="inner"
)

In [71]:
from sklearn.model_selection import train_test_split

train_bal, test_bal = train_test_split(
    places_with_reviews_balanced,
    test_size=0.5,
    random_state=42
)

In [72]:
vectorizer_bal = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    max_df=0.8
)

tfidf_test_bal = vectorizer_bal.fit_transform(test_bal["review"])

In [79]:
query_text = train_places.iloc[1]["review"]

tfidf_query = vectorizer_bal.transform([query_text])

similarities = cosine_similarity(tfidf_query, tfidf_test_bal)[0]

ranking = similarities.argsort()[::-1]

In [80]:
best_index = ranking[0]
best_place = test_places.iloc[best_index]

print("Query type:", train_places.iloc[1]["typeR"])
print("Recommended type:", best_place["typeR"])

Query type: A
Recommended type: A


### Ranking evaluation

In [42]:
errors_level2_balanced = []

for i in tqdm(range(len(train_bal))):
    
    query_row = train_bal.iloc[i]
    query_text = query_row["review"]
    
    tfidf_query = vectorizer_bal.transform([query_text])
    similarities = cosine_similarity(tfidf_query, tfidf_test_bal)[0]
    
    ranking = similarities.argsort()[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_row = test_bal.iloc[idx]
        
        if is_match_level2(query_row, candidate_row):
            error = rank
            break
    
    if error is not None:
        errors_level2_balanced.append(error)

mean_error_balanced = np.mean(errors_level2_balanced)

print("Ranking Error Level 2 (TF-IDF balanced 20 reviews):", mean_error_balanced)
print("Evaluated queries:", len(errors_level2_balanced))

100%|██████████| 917/917 [00:15<00:00, 61.06it/s]

Ranking Error Level 2 (TF-IDF balanced 20 reviews): 2.4873487348734873
Evaluated queries: 909


### 2.6.2 10 reviews (optimum)

10 reviews max here, the best choice.

In [81]:
MAX_REVIEWS = 10

balanced_reviews_10 = (
    reviews_en
    .sort_values("idplace")
    .groupby("idplace")
    .head(MAX_REVIEWS)
)

In [82]:
grouped_balanced_10 = (
    balanced_reviews_10
    .groupby("idplace")["review"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

In [83]:
places_with_reviews_balanced_10 = places.merge(
    grouped_balanced_10,
    left_on="id",
    right_on="idplace",
    how="inner"
)

In [84]:
train_bal_10, test_bal_10 = train_test_split(
    places_with_reviews_balanced_10,
    test_size=0.5,
    random_state=42
)

In [85]:
vectorizer_bal_10 = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    max_df=0.8
)

tfidf_test_bal_10 = vectorizer_bal_10.fit_transform(test_bal_10["review"])

In [86]:
query_text = train_places.iloc[1]["review"]

tfidf_query = vectorizer_bal.transform([query_text])

similarities = cosine_similarity(tfidf_query, tfidf_test_bal)[0]

ranking = similarities.argsort()[::-1]

In [87]:
best_index = ranking[0]
best_place = test_places.iloc[best_index]

print("Query type:", train_places.iloc[1]["typeR"])
print("Recommended type:", best_place["typeR"])

Query type: A
Recommended type: A


### Ranking evaluation

In [38]:
errors_level2_balanced_10 = []

for i in tqdm(range(len(train_bal_10))):
    
    query_row = train_bal_10.iloc[i]
    query_text = query_row["review"]
    
    tfidf_query = vectorizer_bal_10.transform([query_text])
    similarities = cosine_similarity(tfidf_query, tfidf_test_bal_10)[0]
    
    ranking = similarities.argsort()[::-1]
    
    error = None
    
    for rank, idx in enumerate(ranking):
        candidate_row = test_bal_10.iloc[idx]
        
        if is_match_level2(query_row, candidate_row):
            error = rank
            break
    
    if error is not None:
        errors_level2_balanced_10.append(error)

mean_error_balanced_10 = np.mean(errors_level2_balanced_10)

print("Ranking Error Level 2 (TF-IDF balanced 10 reviews):", mean_error_balanced_10)
print("Evaluated queries:", len(errors_level2_balanced_10))

100%|██████████| 917/917 [00:12<00:00, 72.92it/s]

Ranking Error Level 2 (TF-IDF balanced 10 reviews): 2.132013201320132
Evaluated queries: 909


## 3. Conclusion

In [40]:
import pandas as pd

# Création du tableau récapitulatif
results = pd.DataFrame([
    {
        "Model": "BM25",
        "Data Strategy": "All reviews",
        "Level 1 Error": 0.9389,
        "Level 2 Error": 5.6667,
        "Notes": "Probabilistic IR baseline"
    },
    {
        "Model": "TF-IDF (unigrams)",
        "Data Strategy": "All reviews",
        "Level 1 Error": 0.8899,
        "Level 2 Error": 3.7734,
        "Notes": "Basic TF-IDF representation"
    },
    {
        "Model": "TF-IDF (1,2-grams improved)",
        "Data Strategy": "All reviews",
        "Level 1 Error": None,
        "Level 2 Error": 3.3751,
        "Notes": "Bigrams + frequency filtering"
    },
    {
        "Model": "Sentence Embeddings",
        "Data Strategy": "All reviews",
        "Level 1 Error": None,
        "Level 2 Error": 4.2409,
        "Notes": "Transformer-based semantic embeddings"
    },
    {
        "Model": "TF-IDF (1,2-grams improved)",
        "Data Strategy": "Max 20 reviews per place",
        "Level 1 Error": None,
        "Level 2 Error": 2.4873,
        "Notes": "Balanced document length"
    },
    {
        "Model": "TF-IDF (1,2-grams improved)",
        "Data Strategy": "Max 10 reviews per place",
        "Level 1 Error": None,
        "Level 2 Error": 2.1320,
        "Notes": "Best performing configuration"
    }
])

# Arrondir 
results = results.round(4)

# Trier par Level 2 Error
results_sorted = results.sort_values(by="Level 2 Error")

results_sorted

,Model,Data Strategy,Level 1 Error,Level 2 Error,Notes
5,"TF-IDF (1,2-grams improved)",Max 10 reviews per place,NaN,2.1320,Best performing configuration
4,"TF-IDF (1,2-grams improved)",Max 20 reviews per place,NaN,2.4873,Balanced document length
2,"TF-IDF (1,2-grams improved)",All reviews,NaN,3.3751,Bigrams + frequency filtering
1,TF-IDF (unigrams),All reviews,0.8899,3.7734,Basic TF-IDF representation
3,Sentence Embeddings,All reviews,NaN,4.2409,Transformer-based semantic embeddings
0,BM25,All reviews,0.9389,5.6667,Probabilistic IR baseline


In [41]:
from IPython.display import display

display(results_sorted.style.highlight_min(subset=["Level 2 Error"], color="lightgreen"))

,Model,Data Strategy,Level 1 Error,Level 2 Error,Notes
5,"TF-IDF (1,2-grams improved)",Max 10 reviews per place,nan,2.132000,Best performing configuration
4,"TF-IDF (1,2-grams improved)",Max 20 reviews per place,nan,2.487300,Balanced document length
2,"TF-IDF (1,2-grams improved)",All reviews,nan,3.375100,Bigrams + frequency filtering
1,TF-IDF (unigrams),All reviews,0.889900,3.773400,Basic TF-IDF representation
3,Sentence Embeddings,All reviews,nan,4.240900,Transformer-based semantic embeddings
0,BM25,All reviews,0.938900,5.666700,Probabilistic IR baseline
